# Trajectory optimization: shortest reliable path

This Northstar incident lab compares a wasteful successful investigation with a shorter evidence-backed one. The goal is not fewer tokens; it is a safe, grounded result with the least justified work.


![Trajectory comparison](assets/trajectory-comparison.svg)

The README contains an editable Mermaid lifecycle; this static SVG renders reliably in GitHub notebooks.


## 1. Instrument before optimizing

Record model/tool calls, arguments, cache hits, retries, latency, cost, policy blocks, evidence contribution, and terminal outcome. Treat safety, tenant scope, and supported correctness as hard constraints; then optimize steps, cost, and latency.


In [ ]:
from lab import WASTEFUL, OPTIMIZED, measure, release_gate, Step, Trace, choose_parallel

for trace in (WASTEFUL, OPTIMIZED):
    print(measure(trace))
    print('gate:', release_gate(trace))


## 2. Diagnose the waste

The wasteful trace repeats `search_incidents` and `query_logs` without a new evidence gap. Reflection is useful only when it changes a bounded next action; otherwise it becomes an expensive loop. The optimized trace uses health, logs, and a runbook exactly once, then stops when independent evidence is sufficient.


In [ ]:
# Experiment A: removing required evidence makes a shorter but unreliable trace.
too_short = Trace('too-short', (Step('get_status',1100,.005,True),), True)
print(release_gate(too_short))

# Experiment B: parallelism needs independence and downstream capacity.
print('independent reads:', choose_parallel(list(OPTIMIZED.steps[:2]), True, True))
print('rate-limited source:', choose_parallel(list(OPTIMIZED.steps[:2]), True, False))


## 3. Techniques and trade-offs

- **Routing:** pick the smallest eligible tool set using typed task/evidence gaps.
- **Caching:** cache only tenant-scoped, fresh, authorization-safe reads.
- **Parallelism:** fan out independent reads with concurrency/rate limits; never parallelize a dependent operation.
- **Compression:** carry a cited evidence summary, not unbounded transcript history.
- **Budgets:** cap calls, time, retries, tokens, and fan-out; spend more only on a named uncertainty.
- **Recovery:** retries must be classified and idempotent; a timeout is not proof a tool failed.

## 4. Release checklist

Run a frozen baseline and holdout suite. Block regressions in forbidden actions, evidence support, policy compliance, and recovery before comparing cost/latency. Monitor cost per successful compliant task after deployment and add real failures as regression cases.

### Exercises

1. Add an expensive duplicate tool call and calculate marginal cost.
2. Add a cache key with tenant, data version, and authorization scope.
3. Define an information-gain threshold for one additional query.
4. Compare a parallel route against a sequential route under a rate limit.

### References

- [Anthropic: Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)
- [OpenAI evaluation best practices](https://developers.openai.com/api/docs/guides/evaluation-best-practices)
